In [1]:
"""
====================================================================
PROBE STUDIES -- FULL, COMBINED, ALL 7 EXPLAINABLE MODELS
Tissue-Structure Reconstruction for Cross-Centre Generalization
(PID 23 -- Achanta Badrinath / Peesari Sathvik Reddy)

WHAT THIS RUNS, IN ONE STANDALONE SCRIPT
--------------------------------------------------------------------
This merges the three separate probing scripts that existed before
(probe_battery_full.py, centre_signature_probe.py, and
pcam_cross_dataset_probe.py) into ONE run, and extends every probe
that is architecturally meaningful for it to baseline_stain_aug and
the 3 open pathology foundation models (kaiko_vits16, phikon,
lunit_vits16) -- not just the 3 self-supervised reconstruction
backbones (recon_resnet18, recon_vit_small, recon_swin_tiny) they
originally covered.

STUDY A -- REPRESENTATION-ANALYSIS PROBE BATTERY (probes 2-5; probe 1,
the tumor-vs-normal probe, stays DISABLED for every model -- see
"WHY PROBE 1 IS STILL DISABLED" below, an already-settled finding
carried over unchanged from probe_battery_full.py):

  2. CENTRE PROBE (deep features)   -- can a probe tell which hospital
                                         a 3x3 tissue window came from,
                                         from the model's own pooled
                                         window-level representation?
  3. CENTRE TRIVIAL-BASELINE PROBE  -- same question from dumb per-patch
                                         pixel colour statistics alone
                                         (no model). Shared across every
                                         model -- computed once.
  4. SPATIAL-POSITION PROBE         -- given one patch's own embedding
                                         (from a REAL, un-shuffled 3x3
                                         window), can a probe recover
                                         which of the 9 grid cells it
                                         came from?
  5. PATIENT-IDENTITY PROBE         -- closed-set: can a probe tell
                                         which (of the known) patients a
                                         patch came from, random
                                         window-level split.

STUDY B -- CROSS-DATASET GENERALIZATION PROBE (Camelyon17-WILDS ->
PatchCamelyon): a tumor probe fit ONLY on Camelyon17-WILDS features,
evaluated in-domain (held-out Camelyon17-WILDS patients) and on
PatchCamelyon's official test split (a disjoint set of hospitals,
CAMELYON16-derived) -- reports the in-domain ceiling, the cross-dataset
score, and the generalization gap between them, for every model.

--------------------------------------------------------------------
EXTENDING PROBES 2/4/5 AND STUDY B TO baseline_stain_aug / THE 3
FOUNDATION MODELS -- WHAT "ARCHITECTURALLY VALID" MEANS HERE
--------------------------------------------------------------------
The 3 reconstruction backbones have a purpose-built windowed
architecture: 9 patches go in together, a transformer mixes them
(self-attention across slots, positional slot_embed, a learned
mask_token for "unknown neighbour"), and what comes out of slot i is
that patch's representation AFTER seeing its neighbours. That
cross-patch mixing is exactly what probes 2/4/5 are testing for signs
of leaking hospital / grid-position / patient identity.

baseline_stain_aug and the 3 foundation models have NO such mechanism
-- they are plain per-patch feature extractors (a frozen resnet18 /
ViT that processes one 96x96 or 224x224 patch at a time, with no
notion of "neighbour" or "window" at all). So for these 4 models:

  - Probe 2 (centre, deep) uses the MEAN of the 9 patches' *independent*
    per-patch features as the "window representation" (there is no
    context-transformer output to pool instead) -- reported and
    labelled explicitly as "patch-mean-pooled", never conflated with
    the reconstruction backbones' "context-pooled" features.
  - Probe 4 (position) uses each patch's own *independent* feature
    directly (no cross-patch mixing exists to strip out) -- this is
    actually the more informative comparison, not a downgrade: it asks
    "does encoding a patch in total isolation already leak where it
    sits in a 3x3 neighbourhood (e.g. via directional tissue/tumor-
    margin gradients), or does that require real cross-patch context?"
  - Probe 5 (patient) and Study B (cross-dataset) both reduce cleanly
    to "one patch in, one embedding out" with no window construction
    needed at all for these 4 models -- there is nothing here that a
    windowed architecture does differently in principle, so this one
    is a completely fair, apples-to-apples comparison.

Every results table/figure below is by MODEL, and a `representation`
column/annotation says in words which of the two regimes produced that
row, so nothing is silently averaged across an architectural difference.

WHY PROBE 1 IS STILL DISABLED (unchanged finding, not re-investigated
here -- see probe_battery_full.py's original docstring for the full
derivation): Camelyon17-WILDS's patch geometry makes a complete 3x3
window of tumor-MINORITY patches essentially never occur (dense,
confluent tumor regions are what actually forms full 3x3 blocks of
present, contiguous patches; normal tissue is sampled too sparsely
across a slide to ever complete a window) -- so whole-window tumor
probing is structurally broken regardless of which model is plugged
in, and re-deriving that for 4 more models would not teach us anything
new. Study B's cross-dataset tumor probe is unaffected (it draws
single patches directly, no window-completeness requirement at all).

CONFIG
--------------------------------------------------------------------
  - 3 reconstruction backbones: loaded from the SAME kagglehub dataset
    used everywhere else in this project
    ("peesarisathvikreddy/histopathology-reconstruction-checkpoints").
  - baseline_stain_aug: loaded from the SAME per-fold checkpoints
    dataset used by the combined XAI notebook ("setu295452/stain-aug"),
    using the TEST_CENTER=2 held-out fold (the "primary" split, same
    convention as every other script in this project).
  - kaiko_vits16 / phikon / lunit_vits16: loaded directly from the
    Hugging Face Hub (open weights, no login needed) -- identical
    loader code to the combined XAI notebook's build_foundation_model().

Run:
    python probe_studies_full_combined.py
====================================================================
"""

import os
import sys
import math
import random
import pickle
import gc
import glob
import warnings
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    PCAM_HF = "1aurent/PatchCamelyon"
    PCAM_SPLIT = "test"

    RECON_KAGGLEHUB_DATASET = "peesarisathvikreddy/histopathology-reconstruction-checkpoints"
    RECON_CHECKPOINT_DIR = "/kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints"
    RECON_CHECKPOINT_MATCH = {
        'recon_resnet18':  'resnet18',
        'recon_vit_small': 'vit_small',
        'recon_swin_tiny': 'swin_tiny',
    }
    RECON_BACKBONE_OF = {   # maps our model name -> the underlying architecture name
        'recon_resnet18':  'resnet18',
        'recon_vit_small': 'vit_small',
        'recon_swin_tiny': 'swin_tiny',
    }
    RECON_BACKBONE_CONFIGS = {
        'resnet18':  dict(dropout=0.3, freeze_stages=2, context_layers=4),
        'vit_small': dict(dropout=0.1, freeze_stages=0, context_layers=4),
        'swin_tiny': dict(dropout=0.1, freeze_stages=0, context_layers=3),
    }

    STAIN_AUG_KAGGLEHUB_DATASET = "setu295452/stain-aug"
    STAIN_AUG_INPUT_DIR = "/kaggle/input/stain-aug"
    STAIN_AUG_TEST_CENTER = 2   # the "primary" fold -- same convention as every other script here
    STAIN_AUG_INPUT_SIZE = 96

    FOUNDATION_MODELS = {
        'kaiko_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms',
            'loader': 'timm', 'input_size': 224,
        },
        'phikon': {
            'hub_id': 'owkin/phikon',
            'loader': 'transformers', 'input_size': 224,
        },
        'lunit_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.lunit_dino',
            'loader': 'timm', 'input_size': 224,
        },
    }

    # Row order for every combined table/figure below: reconstruction backbones
    # first (the original scope of these probes), then the stain-aug baseline,
    # then the 3 foundation models -- newest additions last.
    MODEL_ORDER = ['recon_resnet18', 'recon_vit_small', 'recon_swin_tiny',
                   'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon']
    WINDOWED_MODELS = {'recon_resnet18', 'recon_vit_small', 'recon_swin_tiny'}   # true context-transformer models
    PATCH_LEVEL_MODELS = {'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon'}

    GRID_SIZE = 3
    NUM_POSITIONS = 9
    NUM_PERMUTATIONS = 500     # unused by these probes, kept only because ReconstructionModel's
                               # __init__ signature expects it (perm_head is never called here)
    CNN_INPUT_SIZE = 96
    VIT_INPUT_SIZE = 224
    COLOR_STAT_SIZE = 64

    ALL_CENTERS = [0, 1, 2, 3, 4]
    SAMPLES_PER_CENTER = 1500       # windows per centre (x9 patches each), Study A
    VAL_PATIENT_FRAC = 0.3
    SEED = 42

    # ---- Study B (cross-dataset) sample sizes ----
    CAMELYON17_LABEL_COLUMN = "label"
    CAMELYON17_TRAIN_N_PER_CLASS = 3000
    CAMELYON17_TEST_N_PER_CLASS = 1000
    PCAM_LABEL_COLUMN = "label"
    PCAM_TEST_N_PER_CLASS = 3000
    LABEL_CANDIDATE_KEYWORDS = ("tumor", "label", "target", "is_tumor", "y")
    SKEW_OVERRIDE_THRESHOLD = 0.97
    BALANCE_DIAGNOSTIC_SAMPLE = 20000
    MIN_CLASS_FOR_RELIABLE_AUROC = 30

    NUM_WORKERS = 4
    CACHE_DIR = "/kaggle/working/cache"
    RESULTS_DIR = "/kaggle/working/probe_results"


os.makedirs(Config.CACHE_DIR, exist_ok=True)
os.makedirs(Config.RESULTS_DIR, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


def get_input_size(name):
    if name in Config.WINDOWED_MODELS:
        backbone = Config.RECON_BACKBONE_OF[name]
        return Config.VIT_INPUT_SIZE if backbone in ('vit_small', 'swin_tiny') else Config.CNN_INPUT_SIZE
    if name == 'baseline_stain_aug':
        return Config.STAIN_AUG_INPUT_SIZE
    return Config.FOUNDATION_MODELS[name]['input_size']


def build_transform(name):
    size = get_input_size(name)
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(), norm])


COLOR_TRANSFORM = transforms.Compose([
    transforms.Resize((Config.COLOR_STAT_SIZE, Config.COLOR_STAT_SIZE)), transforms.ToTensor(),
])


# ====================================================================
# LATTICE / SPATIAL INDEX  (model-agnostic -- built once, shared by
# every model in Study A)
# ====================================================================
class Lattice:
    __slots__ = ('patient', 'node', 'center', 'idx_grid', 'valid_centers')

    def __init__(self, patient, node, center, patches, grid_size):
        self.patient, self.node, self.center = patient, node, center
        xs = sorted({p['x'] for p in patches})
        ys = sorted({p['y'] for p in patches})
        x_to_c = {x: i for i, x in enumerate(xs)}
        y_to_r = {y: i for i, y in enumerate(ys)}
        g = np.full((len(ys), len(xs)), -1, dtype=np.int64)
        for p in patches:
            g[y_to_r[p['y']], x_to_c[p['x']]] = p['idx']
        self.idx_grid = g

        H, W = g.shape
        valid = []
        if H >= grid_size and W >= grid_size:
            filled = g >= 0
            integral = np.zeros((H + 1, W + 1), dtype=np.int64)
            integral[1:, 1:] = filled.astype(np.int64).cumsum(0).cumsum(1)
            for r in range(0, H - grid_size + 1):
                for c in range(0, W - grid_size + 1):
                    s = (integral[r + grid_size, c + grid_size]
                         - integral[r, c + grid_size]
                         - integral[r + grid_size, c]
                         + integral[r, c])
                    if s == grid_size * grid_size:
                        valid.append((r, c))
        self.valid_centers = np.array(valid, dtype=np.int32) if valid else np.zeros((0, 2), dtype=np.int32)

    def window(self, r, c, grid_size):
        return self.idx_grid[r:r + grid_size, c:c + grid_size].ravel()


class SpatialIndex:
    def __init__(self, dataset, grid_size, cache_path=None):
        self.grid_size = grid_size
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                self.lattices = pickle.load(f)
            print(f"OK loaded {len(self.lattices)} lattices from cache")
        else:
            self.lattices = self._build(dataset)
            if cache_path:
                with open(cache_path, 'wb') as f:
                    pickle.dump(self.lattices, f)
                print(f"OK cached -> {cache_path}")

    def _build(self, dataset):
        print("Building lattices...")
        meta = dataset.remove_columns(['image']).to_pandas()
        groups = defaultdict(list)
        for row in tqdm(meta.itertuples(index=True), total=len(meta), desc="index"):
            groups[(int(row.patient), int(row.node), int(row.center))].append(
                {'x': int(row.x_coord), 'y': int(row.y_coord), 'idx': int(row.Index)})
        lattices = []
        for (pat, node, ctr), patches in tqdm(groups.items(), desc="lattices"):
            lat = Lattice(pat, node, ctr, patches, self.grid_size)
            if len(lat.valid_centers):
                lattices.append(lat)
        return lattices

    def subset(self, centers, patients=None):
        pats = set(patients) if patients is not None else None
        return [lat for lat in self.lattices
                if lat.center in centers and (pats is None or lat.patient in pats)]


# ====================================================================
# MODEL A -- the windowed ReconstructionModel (recon_resnet18/vit_small/
# swin_tiny). Real cross-patch mixing via a transformer encoder over the
# 9 slots, positional slot_embed, and a learned mask_token for "unknown
# neighbour" -- identical architecture/loading to every other probe
# script in this project (probe_battery_full.py, centre_signature_probe.py,
# pcam_cross_dataset_probe.py), so numbers here are directly comparable
# to any of those.
# ====================================================================
class ReconstructionModel(nn.Module):
    def __init__(self, backbone_name, num_positions=9, num_permutations=500,
                 dropout=0.1, freeze_stages=0, context_layers=4):
        super().__init__()
        self.backbone_name = backbone_name
        self.num_positions = num_positions
        self.input_size = get_input_size(
            {'resnet18': 'recon_resnet18', 'vit_small': 'recon_vit_small',
             'swin_tiny': 'recon_swin_tiny'}[backbone_name])
        self._build_backbone(backbone_name)
        self.proj = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.LayerNorm(self.feature_dim),
        )
        self.context = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=self.feature_dim, nhead=8,
                dim_feedforward=min(4 * self.feature_dim, 2048),
                dropout=dropout, batch_first=True, norm_first=True),
            num_layers=context_layers)
        self.slot_embed = nn.Parameter(torch.zeros(1, num_positions, self.feature_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.feature_dim))
        # perm_head/position_head exist in the trained checkpoint's state_dict
        # (so load_state_dict has somewhere to put those weights) but are never
        # called by any probe here -- probes 2/4/5 all read off get_embedding()'s
        # context-transformer output directly, the same as every other probe
        # script in this project.
        self.perm_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim * num_positions, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_permutations))
        self.position_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_positions))

    def _build_backbone(self, name):
        if name == 'resnet18':
            net = tvm.resnet18(weights=None)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)
        elif name == 'vit_small':
            import timm
            self.backbone = timm.create_model(
                'vit_small_patch16_224', pretrained=False,
                num_classes=0, img_size=self.input_size)
            self.feature_dim, self.pool = 384, None
        elif name == 'swin_tiny':
            import timm
            self.backbone = timm.create_model(
                'swin_tiny_patch4_window7_224', pretrained=False,
                num_classes=0, img_size=self.input_size)
            self.feature_dim, self.pool = 768, None
        else:
            raise ValueError(f"Backbone {name} not supported in this file")

    def _encode(self, x, mask=None):
        B, N = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        f = self.proj(f).view(B, N, self.feature_dim)
        if mask is not None and mask.any():
            f = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(f), f)
        return self.context(f + self.slot_embed)

    @torch.no_grad()
    def get_embedding(self, x, mask=None):
        return self._encode(x, mask=mask)


def _resolve_recon_checkpoint_dir():
    alt_paths = [
        Config.RECON_CHECKPOINT_DIR,
        "/kaggle/input/histopathology-reconstruction-checkpoints",
    ]
    for p in alt_paths:
        if os.path.isdir(p):
            print(f"OK found recon checkpoints at classic input mount {p}")
            return p
    import kagglehub
    path = kagglehub.dataset_download(Config.RECON_KAGGLEHUB_DATASET)
    print(f"OK resolved recon checkpoints via kagglehub -> {path}")
    return path


def _find_recon_checkpoints(root):
    all_pt = glob.glob(os.path.join(root, '**', '*.pt'), recursive=True)
    found = {}
    for name, needle in Config.RECON_CHECKPOINT_MATCH.items():
        cand = [p for p in all_pt if needle in os.path.basename(p).lower()]
        if cand:
            found[name] = cand[0]
        else:
            print(f"FLAG no checkpoint matched for '{name}' (looked for '*{needle}*.pt' under {root})")
    return found


def load_recon_model(name, ckpt_path, device):
    backbone = Config.RECON_BACKBONE_OF[name]
    cfg = Config.RECON_BACKBONE_CONFIGS[backbone]
    model = ReconstructionModel(
        backbone, Config.NUM_POSITIONS, Config.NUM_PERMUTATIONS,
        dropout=cfg['dropout'], freeze_stages=cfg['freeze_stages'],
        context_layers=cfg['context_layers'],
    ).to(device)
    # weights_only=False: PyTorch 2.6 flipped torch.load's default to True, which
    # rejects the plain numpy scalars (e.g. best_pos) these checkpoints also carry
    # alongside the state_dict ("Unsupported global: numpy._core.multiarray.scalar").
    # These are our own checkpoints, so this is safe.
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"FLAG [{name}] load_state_dict mismatches -- missing={len(missing)} unexpected={len(unexpected)}")
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f"OK [{name}] loaded {ckpt_path}")
    model.feature_dim_out = model.feature_dim
    return model


# ====================================================================
# MODEL B -- PATCH-LEVEL ADAPTER (baseline_stain_aug + the 3 foundation
# models). These process ONE patch at a time, independently -- no
# cross-patch mixing exists, so `get_embedding` on a window just encodes
# each of the 9 real patches on its own, and under the masked-single-
# patch protocol (mask is not None) it short-circuits to encoding ONLY
# slot 0 (the one real patch) -- every caller of get_embedding under
# that protocol only ever reads back slot 0 anyway (see extract_masked_*
# below), so skipping slots 1-8 entirely is a pure speed win, not an
# approximation.
# ====================================================================
class PatchLevelAdapter(nn.Module):
    def __init__(self, encoder, feature_dim):
        super().__init__()
        self.encoder = encoder      # any nn.Module exposing .encode_single(x) -> (B, D)
        self.feature_dim = feature_dim

    @torch.no_grad()
    def get_embedding(self, x, mask=None):
        B, N = x.shape[:2]
        if mask is not None and mask.any():
            real = x[:, 0]                                  # (B, C, H, W) -- slot 0 only
            f0 = self.encoder.encode_single(real)
            out = torch.zeros(B, N, self.feature_dim, device=x.device, dtype=f0.dtype)
            out[:, 0] = f0
            return out
        flat = x.flatten(0, 1)                               # (B*N, C, H, W)
        f = self.encoder.encode_single(flat)
        return f.view(B, N, self.feature_dim)


def build_stain_aug_classifier():
    """Identical architecture to the combined XAI notebook's version -- an
    ImageNet-initialized resnet18 with a fresh binary head, trained end-to-end."""
    net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(net.fc.in_features, 1)
    return net


class _StainAugEncoder(nn.Module):
    """Strips the trained classification head off a loaded baseline_stain_aug
    checkpoint, keeping everything through avgpool -- i.e. the same 512-dim
    penultimate representation every other probe in this project reads off
    a resnet18-family model."""
    def __init__(self, stain_aug_model):
        super().__init__()
        self.backbone = nn.Sequential(*list(stain_aug_model.children())[:-1])   # drops fc only

    @torch.no_grad()
    def encode_single(self, x):
        f = self.backbone(x)
        return torch.flatten(f, 1)


def _resolve_stain_aug_checkpoint():
    alt_paths = [Config.STAIN_AUG_INPUT_DIR]
    root = None
    for p in alt_paths:
        if os.path.isdir(p):
            root = p
            print(f"OK found stain-aug checkpoints at classic input mount {p}")
            break
    if root is None:
        import kagglehub
        root = kagglehub.dataset_download(Config.STAIN_AUG_KAGGLEHUB_DATASET)
        print(f"OK resolved stain-aug checkpoints via kagglehub -> {root}")

    target = f"stain_aug_holdout{Config.STAIN_AUG_TEST_CENTER}.pt"
    matches = glob.glob(os.path.join(root, '**', target), recursive=True)
    if not matches:
        raise FileNotFoundError(f"could not find {target} under {root}")
    return matches[0]


def load_stain_aug_adapter(device):
    ckpt_path = _resolve_stain_aug_checkpoint()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)   # see FIX note above
    net = build_stain_aug_classifier().to(device)
    net.load_state_dict(ckpt['state_dict'])
    net.eval()
    for p in net.parameters():
        p.requires_grad = False
    print(f"OK [baseline_stain_aug] loaded {ckpt_path} "
          f"(trained on centres {ckpt['train_centers']}, primary AUROC={ckpt['auroc']:.4f})")
    adapter = PatchLevelAdapter(_StainAugEncoder(net).to(device), feature_dim=512)
    return adapter


class _TimmViTEncoder(nn.Module):
    def __init__(self, hub_id):
        super().__init__()
        import timm as _timm
        self.backbone = _timm.create_model(hub_id, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, 'num_features', None) or 768

    @torch.no_grad()
    def encode_single(self, x):
        f = self.backbone(x)
        return f if f.dim() == 2 else f.mean(1)


class _HFTransformersViTEncoder(nn.Module):
    def __init__(self, hub_id):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(hub_id, attn_implementation="eager")
        self.feature_dim = self.backbone.config.hidden_size

    @torch.no_grad()
    def encode_single(self, x):
        out = self.backbone(pixel_values=x)
        return out.last_hidden_state[:, 0]   # CLS token


def load_foundation_adapter(name, device):
    cfg = Config.FOUNDATION_MODELS[name]
    if cfg['loader'] == 'transformers':
        enc = _HFTransformersViTEncoder(cfg['hub_id']).to(device)
    else:
        enc = _TimmViTEncoder(cfg['hub_id']).to(device)
    enc.eval()
    for p in enc.parameters():
        p.requires_grad = False
    print(f"OK [{name}] loaded {cfg['hub_id']} (dim={enc.feature_dim})")
    return PatchLevelAdapter(enc, feature_dim=enc.feature_dim)


# ====================================================================
# UNIFIED MODEL LOADER -- returns an object exposing .get_embedding(x,
# mask=None) and .feature_dim, regardless of which of the two model
# families (windowed vs. patch-level) it actually is. Every probe below
# is written against this ONE interface.
# ====================================================================
def load_model_for_probes(name, device, recon_ckpts):
    if name in Config.WINDOWED_MODELS:
        return load_recon_model(name, recon_ckpts[name], device)
    if name == 'baseline_stain_aug':
        return load_stain_aug_adapter(device)
    if name in Config.FOUNDATION_MODELS:
        return load_foundation_adapter(name, device)
    raise ValueError(f"no loader registered for {name}")


# ====================================================================
# DATA LOADING
# ====================================================================
def load_camelyon17():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    print(f"[Camelyon17-WILDS] splits: {', '.join(f'{k}={len(ds[k]):,}' for k in ds.keys())}")
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"OK [Camelyon17-WILDS] concatenated: {len(full):,} patches")
    return full


def load_pcam():
    from datasets import load_dataset
    ds = load_dataset(Config.PCAM_HF)
    print(f"[PatchCamelyon] splits: {', '.join(f'{k}={len(ds[k]):,}' for k in ds.keys())}")
    if Config.PCAM_SPLIT not in ds:
        raise ValueError(f"'{Config.PCAM_SPLIT}' split not found in {Config.PCAM_HF}; available: {list(ds.keys())}")
    split = ds[Config.PCAM_SPLIT]
    print(f"OK [PatchCamelyon] using split '{Config.PCAM_SPLIT}': {len(split):,} patches "
          f"(official held-out split -- Study B never trains on this)")
    return split


def resolve_label_column(dataset, dataset_name, configured_col):
    """Shared label-column auto-detection (unchanged logic from
    probe_battery_full.py / pcam_cross_dataset_probe.py): checks that the
    configured label column isn't severely skewed before trusting it."""
    non_image_cols = [c for c in dataset.column_names if c != 'image']
    candidates = [c for c in non_image_cols
                  if any(k in c.lower() for k in Config.LABEL_CANDIDATE_KEYWORDS)]
    if configured_col not in candidates and configured_col in non_image_cols:
        candidates.append(configured_col)

    print(f"\n--- LABEL COLUMN DIAGNOSTIC -- {dataset_name} ---")
    print(f"candidates: {candidates or '(none -- keeping configured default)'}")
    if not candidates:
        return configured_col

    n = min(Config.BALANCE_DIAGNOSTIC_SAMPLE, len(dataset))
    rng = np.random.RandomState(Config.SEED)
    idx = sorted(rng.choice(len(dataset), size=n, replace=False).tolist())
    sub = dataset.select(idx).select_columns(candidates).to_pandas()

    stats = {}
    for col in candidates:
        vals = sub[col]
        uniq = sorted(vals.unique().tolist())
        if len(uniq) != 2:
            continue
        pos_rate = float((vals == uniq[-1]).mean())
        skew = max(pos_rate, 1 - pos_rate)
        stats[col] = skew
        flag = "  <-- SEVERELY SKEWED" if skew > Config.SKEW_OVERRIDE_THRESHOLD else ""
        print(f"   '{col}': positive-rate={pos_rate:.4f} (skew={skew:.4f}){flag}")

    if not stats:
        return configured_col
    configured_skew = stats.get(configured_col, 1.0)
    best_col = min(stats, key=stats.get)
    best_skew = stats[best_col]
    if configured_skew > Config.SKEW_OVERRIDE_THRESHOLD and best_skew < configured_skew:
        print(f"FLAG switching label column '{configured_col}' -> '{best_col}'")
        return best_col
    return configured_col


def build_patient_split_by_centre(camelyon17_meta, val_frac, seed):
    """Per-centre, patient-level split -- identical convention to every other
    script in this project, so numbers here are directly comparable."""
    train_pats, test_pats = {}, {}
    for centre in Config.ALL_CENTERS:
        pats = sorted(camelyon17_meta.loc[camelyon17_meta['center'] == centre, 'patient'].unique().tolist())
        rng_c = random.Random(seed + centre)
        rng_c.shuffle(pats)
        if len(pats) < 2:
            train_pats[centre], test_pats[centre] = set(pats), set()
            continue
        n_test = max(1, int(round(val_frac * len(pats))))
        n_test = min(n_test, len(pats) - 1)
        test_pats[centre] = set(pats[:n_test])
        train_pats[centre] = set(pats[n_test:])
    return train_pats, test_pats


# ====================================================================
# STUDY A -- DATASET: real, un-shuffled 3x3 windows (slot i == true grid
# position i), plus per-patch colour stats and tumor label (used only
# by probe 3's trivial baseline / bookkeeping, not by probes 2/4/5).
# ====================================================================
class CleanWindowDataset(Dataset):
    def __init__(self, hf_dataset, lattices, transform, n_samples, grid_size, seed, label_col):
        self.ds = hf_dataset
        self.lattices = lattices
        self.transform = transform
        self.grid_size = grid_size
        self.label_col = label_col
        rng = np.random.RandomState(seed)
        counts = np.array([len(l.valid_centers) for l in lattices], dtype=np.float64)
        p = counts / counts.sum()
        self.items = []
        for _ in range(n_samples):
            li = int(rng.choice(len(lattices), p=p))
            lat = lattices[li]
            ci = int(rng.randint(len(lat.valid_centers)))
            self.items.append((li, ci))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        li, ci = self.items[i]
        lat = self.lattices[li]
        r, c = lat.valid_centers[ci]
        flat = lat.window(int(r), int(c), self.grid_size)
        tiles, colors, tumor = [], [], []
        for k in flat:
            row = self.ds[int(k)]
            img = row['image']
            img = img.convert('RGB') if img.mode != 'RGB' else img
            tiles.append(self.transform(img))
            cstat = COLOR_TRANSFORM(img)
            mean = cstat.mean(dim=(1, 2))
            std = cstat.std(dim=(1, 2))
            colors.append(torch.cat([mean, std]))
            tumor.append(int(row[self.label_col]))
        return {
            'images': torch.stack(tiles), 'colors': torch.stack(colors),
            'tumor': torch.tensor(tumor, dtype=torch.long),
            'center': lat.center, 'patient': lat.patient,
        }


def collate_window(batch):
    images = torch.stack([b['images'] for b in batch])
    colors = torch.stack([b['colors'] for b in batch])
    tumor = torch.stack([b['tumor'] for b in batch])
    centers = torch.tensor([b['center'] for b in batch], dtype=torch.long)
    patients = torch.tensor([b['patient'] for b in batch], dtype=torch.long)
    return {'images': images, 'colors': colors, 'tumor': tumor, 'center': centers, 'patient': patients}


@torch.no_grad()
def extract_records(model, loader, device):
    """Model-agnostic: works identically whether `model` is a windowed
    ReconstructionModel (get_embedding mixes the 9 slots via self-attention)
    or a PatchLevelAdapter (get_embedding just encodes each of the 9 real
    patches independently, no mixing) -- the caller (and every probe below)
    doesn't need to know which one it got."""
    embs, colors, tumors, positions, centers, patients = [], [], [], [], [], []
    win_embs, win_centers, win_patients = [], [], []
    for batch in tqdm(loader, desc="extract", leave=False):
        x = batch['images'].to(device, non_blocking=True)
        ctx = model.get_embedding(x)                      # (B, 9, D)
        B, N, D = ctx.shape

        embs.append(ctx.reshape(B * N, D).cpu().numpy())
        colors.append(batch['colors'].reshape(B * N, -1).numpy())
        tumors.append(batch['tumor'].reshape(B * N).numpy())
        pos = torch.arange(N).unsqueeze(0).expand(B, N).reshape(-1).numpy()
        positions.append(pos)
        centers.append(batch['center'].repeat_interleave(N).numpy())
        patients.append(batch['patient'].repeat_interleave(N).numpy())

        win_embs.append(ctx.mean(dim=1).cpu().numpy())      # (B, D) -- mean-pooled window feature
        win_centers.append(batch['center'].numpy())
        win_patients.append(batch['patient'].numpy())
    return {
        'emb': np.concatenate(embs), 'color': np.concatenate(colors),
        'tumor': np.concatenate(tumors), 'position': np.concatenate(positions),
        'center': np.concatenate(centers), 'patient': np.concatenate(patients),
        'win_emb': np.concatenate(win_embs), 'win_center': np.concatenate(win_centers),
        'win_patient': np.concatenate(win_patients),
    }


def merge(records_list):
    if not records_list:
        return None
    keys = records_list[0].keys()
    return {k: np.concatenate([r[k] for r in records_list]) for k in keys}


def linear_probe_acc(X_train, y_train, X_test, y_test, max_iter=2000):
    clf = LogisticRegression(max_iter=max_iter)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    return accuracy_score(y_test, pred) * 100


# ====================================================================
# STUDY A -- probes 2 (centre/deep), 3 (centre/trivial-colour), 4
# (position), 5 (patient identity), run once per model.
# ====================================================================
def run_study_a_for_model(name, model, hf, index, device, label_col):
    print("\n" + "=" * 70)
    print(f"STUDY A -- {name}  ({'context-pooled window model' if name in Config.WINDOWED_MODELS else 'patch-independent model'})")
    print("=" * 70)
    transform = build_transform(name)

    n_train_target = int(round(Config.SAMPLES_PER_CENTER * (1 - Config.VAL_PATIENT_FRAC)))
    n_test_target = Config.SAMPLES_PER_CENTER - n_train_target

    train_records, test_records = [], []
    for centre in Config.ALL_CENTERS:
        lats = index.subset([centre])
        if not lats:
            print(f"FLAG no valid windows for centre {centre}, skipping")
            continue
        patients_here = sorted({lat.patient for lat in lats})
        rng_c = random.Random(Config.SEED + centre)
        rng_c.shuffle(patients_here)
        if len(patients_here) < 2:
            train_pats, test_pats = set(patients_here), set()
        else:
            n_test_pats = max(1, int(round(Config.VAL_PATIENT_FRAC * len(patients_here))))
            n_test_pats = min(n_test_pats, len(patients_here) - 1)
            test_pats = set(patients_here[:n_test_pats])
            train_pats = set(patients_here[n_test_pats:])

        lats_train = index.subset([centre], train_pats)
        lats_test = index.subset([centre], test_pats) if test_pats else []

        ds_tr = CleanWindowDataset(hf, lats_train, transform, n_train_target, Config.GRID_SIZE,
                                   seed=Config.SEED + centre, label_col=label_col)
        loader_tr = DataLoader(ds_tr, batch_size=24, shuffle=False,
                               num_workers=Config.NUM_WORKERS, collate_fn=collate_window)
        rec_tr = extract_records(model, loader_tr, device)
        train_records.append(rec_tr)

        n_te = 0
        if lats_test:
            ds_te = CleanWindowDataset(hf, lats_test, transform, n_test_target, Config.GRID_SIZE,
                                       seed=Config.SEED + 1000 + centre, label_col=label_col)
            loader_te = DataLoader(ds_te, batch_size=24, shuffle=False,
                                   num_workers=Config.NUM_WORKERS, collate_fn=collate_window)
            rec_te = extract_records(model, loader_te, device)
            test_records.append(rec_te)
            n_te = len(rec_te['emb'])

        print(f"  centre {centre}: {len(patients_here)} patients "
              f"({len(train_pats)} train / {len(test_pats)} test) -> "
              f"{len(rec_tr['emb'])} train patches, {n_te} test patches")

    results = {}
    train_pool = merge(train_records)
    test_pool = merge(test_records)

    # ---- Probe 2: centre, deep features ----
    chance2 = 100.0 / len(set(train_pool['win_center'].tolist()) | set(test_pool['win_center'].tolist()))
    clf2 = LogisticRegression(max_iter=2000)
    clf2.fit(train_pool['win_emb'], train_pool['win_center'])
    acc2_logreg = accuracy_score(test_pool['win_center'], clf2.predict(test_pool['win_emb'])) * 100
    mlp2 = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=Config.SEED)
    mlp2.fit(train_pool['win_emb'], train_pool['win_center'])
    acc2_mlp = accuracy_score(test_pool['win_center'], mlp2.predict(test_pool['win_emb'])) * 100
    print(f"  [Probe 2 -- centre/deep]  logreg {acc2_logreg:6.2f}%   mlp {acc2_mlp:6.2f}%   (chance {chance2:.2f}%)")
    results['centre_deep_logreg'] = acc2_logreg
    results['centre_deep_mlp'] = acc2_mlp
    results['centre_deep_chance'] = chance2

    # ---- Probe 3: centre, trivial colour baseline (model-independent) ----
    n_centres_present = len(set(train_pool['center'].tolist()) | set(test_pool['center'].tolist()))
    chance3 = 100.0 / n_centres_present
    acc3 = linear_probe_acc(train_pool['color'], train_pool['center'], test_pool['color'], test_pool['center'])
    print(f"  [Probe 3 -- centre/color] acc {acc3:6.2f}%   (chance {chance3:.2f}%)  -- no model, pixel stats only")
    results['centre_trivial'] = acc3
    results['centre_trivial_chance'] = chance3

    # ---- Probe 4: true spatial position (0-8) ----
    chance4 = 100.0 / Config.NUM_POSITIONS
    acc4 = linear_probe_acc(train_pool['emb'], train_pool['position'], test_pool['emb'], test_pool['position'])
    print(f"  [Probe 4 -- position]     acc {acc4:6.2f}%   (chance {chance4:.2f}%)")
    results['position'] = acc4
    results['position_chance'] = chance4

    # ---- Probe 5: patient identity, closed-set, random window-level split ----
    full_pool = merge(train_records + test_records)
    n = len(full_pool['emb'])
    rng = np.random.RandomState(Config.SEED)
    perm = rng.permutation(n)
    split = int(0.7 * n)
    tr_idx, te_idx = perm[:split], perm[split:]
    n_patients = len(set(full_pool['patient'].tolist()))
    chance5 = 100.0 / n_patients
    acc5 = linear_probe_acc(full_pool['emb'][tr_idx], full_pool['patient'][tr_idx],
                            full_pool['emb'][te_idx], full_pool['patient'][te_idx])
    print(f"  [Probe 5 -- patient ID]   acc {acc5:6.2f}%   (chance {chance5:.2f}%, {n_patients} patients, closed-set)")
    results['patient'] = acc5
    results['patient_chance'] = chance5

    return results


# ====================================================================
# STUDY B -- CROSS-DATASET GENERALIZATION (Camelyon17-WILDS -> PatchCamelyon)
# ====================================================================
class SinglePatchDataset(Dataset):
    """One standalone patch + its label. For recon/windowed models this
    deliberately IGNORES real neighbours (masked-single-patch protocol, via
    extract_masked_embeddings below); for patch-level models it's simply
    what they always do -- one patch in, one embedding out. Works
    identically for both Camelyon17-WILDS and PatchCamelyon (which has no
    neighbour/coordinate metadata at all)."""
    def __init__(self, hf_dataset, indices, transform, label_col):
        self.ds = hf_dataset
        self.indices = indices
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.ds[int(self.indices[i])]
        img = row['image']
        img = img.convert('RGB') if img.mode != 'RGB' else img
        return {'image': self.transform(img), 'label': int(row[self.label_col])}


def collate_single(batch):
    images = torch.stack([b['image'] for b in batch])
    labels = torch.tensor([b['label'] for b in batch], dtype=torch.long)
    return {'image': images, 'label': labels}


def stratified_sample_indices(labels, n_per_class, seed):
    rng = np.random.RandomState(seed)
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    rng.shuffle(pos_idx); rng.shuffle(neg_idx)
    k = min(n_per_class, len(pos_idx), len(neg_idx))
    if k < n_per_class:
        print(f"   FLAG only {len(pos_idx)} pos / {len(neg_idx)} neg available -- using {k}/class")
    idx = np.concatenate([pos_idx[:k], neg_idx[:k]])
    rng.shuffle(idx)
    return idx


@torch.no_grad()
def extract_masked_embeddings(model, loader, device):
    """MASKED SINGLE-PATCH PROTOCOL (unchanged rationale from
    pcam_cross_dataset_probe.py): every patch, from EITHER dataset, is
    embedded via the exact same procedure, so a score difference reflects
    dataset shift, not 'windowed context vs. no context'. For a windowed
    model this genuinely masks 8 of the 9 slots with its own mask_token;
    for a PatchLevelAdapter, get_embedding already short-circuits to
    encoding only the one real patch (see PatchLevelAdapter docstring)."""
    embs, labels = [], []
    for batch in tqdm(loader, desc="extract(masked)", leave=False):
        x = batch['image'].to(device, non_blocking=True)       # (B, 3, H, W)
        B = x.shape[0]
        x_win = x.unsqueeze(1).expand(B, Config.NUM_POSITIONS, *x.shape[1:]).contiguous()
        mask = torch.zeros(B, Config.NUM_POSITIONS, dtype=torch.bool, device=device)
        mask[:, 1:] = True
        ctx = model.get_embedding(x_win, mask=mask)             # (B, 9, D)
        embs.append(ctx[:, 0, :].cpu().numpy())
        labels.append(batch['label'].numpy())
    return np.concatenate(embs), np.concatenate(labels)


def tumor_probe_fit(X_train, y_train, X_test, y_test, max_iter=1000):
    clf = LogisticRegression(max_iter=max_iter, class_weight='balanced')
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, pred) * 100
    bal_acc = balanced_accuracy_score(y_test, pred) * 100
    n_errors = int((pred != y_test).sum())
    n_test_pos = int(np.sum(y_test == 1))
    n_test_neg = int(np.sum(y_test == 0))
    auroc_reliable = min(n_test_pos, n_test_neg) >= Config.MIN_CLASS_FOR_RELIABLE_AUROC
    try:
        auroc = roc_auc_score(y_test, proba)
    except ValueError:
        auroc = float('nan')
    return {
        'acc': acc, 'balanced_acc': bal_acc, 'auroc': auroc, 'auroc_reliable': auroc_reliable,
        'n_errors': n_errors, 'n_test': len(y_test),
        'n_test_pos': n_test_pos, 'n_test_neg': n_test_neg,
    }, clf


def tumor_probe_eval(clf, X, y):
    pred = clf.predict(X)
    proba = clf.predict_proba(X)[:, 1]
    acc = accuracy_score(y, pred) * 100
    bal_acc = balanced_accuracy_score(y, pred) * 100
    n_errors = int((pred != y).sum())
    n_pos = int(np.sum(y == 1)); n_neg = int(np.sum(y == 0))
    auroc_reliable = min(n_pos, n_neg) >= Config.MIN_CLASS_FOR_RELIABLE_AUROC
    try:
        auroc = roc_auc_score(y, proba)
    except ValueError:
        auroc = float('nan')
    return {
        'acc': acc, 'balanced_acc': bal_acc, 'auroc': auroc, 'auroc_reliable': auroc_reliable,
        'n_errors': n_errors, 'n_test': len(y), 'n_test_pos': n_pos, 'n_test_neg': n_neg,
    }


def fmt_metric(m):
    auroc_str = f"{m['auroc']:.4f}" + ("" if m['auroc_reliable'] else "*")
    return (f"acc {m['acc']:6.2f}%  balanced-acc {m['balanced_acc']:6.2f}%  auroc {auroc_str:>7}  "
            f"({m['n_errors']}/{m['n_test']} errors, {m['n_test_pos']} pos/{m['n_test_neg']} neg)")


def run_study_b_for_model(name, model, cam17, pcam, device,
                          cam17_label_col, pcam_label_col,
                          train_sel, test_sel, pcam_sel):
    print("\n" + "=" * 70)
    print(f"STUDY B -- {name}  (cross-dataset: Camelyon17-WILDS -> PatchCamelyon)")
    print("=" * 70)
    transform = build_transform(name)

    ds_train = SinglePatchDataset(cam17, train_sel, transform, cam17_label_col)
    ds_indomain = SinglePatchDataset(cam17, test_sel, transform, cam17_label_col)
    ds_pcam = SinglePatchDataset(pcam, pcam_sel, transform, pcam_label_col)

    loader_train = DataLoader(ds_train, batch_size=32, shuffle=False,
                              num_workers=Config.NUM_WORKERS, collate_fn=collate_single)
    loader_indomain = DataLoader(ds_indomain, batch_size=32, shuffle=False,
                                 num_workers=Config.NUM_WORKERS, collate_fn=collate_single)
    loader_pcam = DataLoader(ds_pcam, batch_size=32, shuffle=False,
                             num_workers=Config.NUM_WORKERS, collate_fn=collate_single)

    X_train, y_train = extract_masked_embeddings(model, loader_train, device)
    X_indomain, y_indomain = extract_masked_embeddings(model, loader_indomain, device)
    X_pcam, y_pcam = extract_masked_embeddings(model, loader_pcam, device)

    metrics_indomain, clf = tumor_probe_fit(X_train, y_train, X_indomain, y_indomain)
    metrics_pcam = tumor_probe_eval(clf, X_pcam, y_pcam)
    gap_acc = metrics_indomain['balanced_acc'] - metrics_pcam['balanced_acc']

    print(f"  [in-domain -- Camelyon17->Camelyon17]   {fmt_metric(metrics_indomain)}")
    print(f"  [cross-dataset -- Camelyon17->PCam]     {fmt_metric(metrics_pcam)}")
    print(f"  [generalization gap, balanced-acc]      {gap_acc:+.2f} pts (smaller = better transfer)")

    return {'indomain': metrics_indomain, 'pcam': metrics_pcam, 'gap_balanced_acc': gap_acc}


# ====================================================================
# COMBINED SUMMARY -- tables + figures across all 7 models
# ====================================================================
def print_study_a_summary(all_results_a):
    print("\n" + "=" * 70)
    print("STUDY A SUMMARY -- centre(deep) / centre(colour) / position / patient, all 7 models")
    print("(tumor-vs-normal probe stays disabled for every model -- see module docstring)")
    print("=" * 70)
    print(f"{'model':<20} {'representation':<24} {'centre(deep)':>13} {'centre(color)':>14} "
          f"{'position':>10} {'patient':>10}")
    for name in Config.MODEL_ORDER:
        if name not in all_results_a:
            continue
        r = all_results_a[name]
        rep = "context-pooled (window)" if name in Config.WINDOWED_MODELS else "patch-mean-pooled (indep.)"
        print(f"{name:<20} {rep:<24} {r['centre_deep_logreg']:>12.2f}% {r['centre_trivial']:>13.2f}% "
              f"{r['position']:>9.2f}% {r['patient']:>9.2f}%")
    print(f"\nchance levels -- centre(deep/colour): ~20%   position: 11.11%   patient: varies (closed-set)")


def print_study_b_summary(all_results_b):
    print("\n" + "=" * 70)
    print("STUDY B SUMMARY -- cross-dataset generalization, all 7 models")
    print("=" * 70)
    print(f"{'model':<20} {'in-domain bal-acc':>18} {'PCam bal-acc':>13} {'gap':>8} "
          f"{'in-domain auroc':>16} {'PCam auroc':>11}")
    for name in Config.MODEL_ORDER:
        if name not in all_results_b:
            continue
        r = all_results_b[name]
        id_auroc = f"{r['indomain']['auroc']:.4f}" + ("" if r['indomain']['auroc_reliable'] else "*")
        pc_auroc = f"{r['pcam']['auroc']:.4f}" + ("" if r['pcam']['auroc_reliable'] else "*")
        print(f"{name:<20} {r['indomain']['balanced_acc']:>17.2f}% {r['pcam']['balanced_acc']:>12.2f}% "
              f"{r['gap_balanced_acc']:>+7.2f} {id_auroc:>16} {pc_auroc:>11}")
    print("\n* = AUROC computed from too few minority-class test examples to be reliable.")
    print("gap = in-domain balanced-acc minus PCam balanced-acc; smaller (closer to 0, or negative) is better transfer.")


def make_combined_figures(all_results_a, all_results_b, timestamp):
    names = [n for n in Config.MODEL_ORDER if n in all_results_a]

    # ---- Study A: 4-panel bar chart ----
    metrics = [('centre_deep_logreg', None, 'Centre (deep features)'),
               ('centre_trivial', None, 'Centre (colour-only baseline)'),
               ('position', 100.0 / Config.NUM_POSITIONS, 'Spatial position (0-8)'),
               ('patient', None, 'Patient identity (closed-set)')]
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 4.8))
    colors_bar = ['#4C72B0' if n in Config.WINDOWED_MODELS else '#DD8452' for n in names]
    for ax, (key, fixed_chance, title) in zip(axes, metrics):
        vals = [all_results_a[n][key] for n in names]
        ax.bar(names, vals, color=colors_bar)
        if fixed_chance is not None:
            ax.axhline(fixed_chance, ls='--', c='grey', label=f'chance ({fixed_chance:.1f}%)')
            ax.legend(fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.set_ylabel('accuracy (%)')
        ax.tick_params(axis='x', rotation=35)
        ax.grid(alpha=.3, axis='y')
    fig.suptitle("Study A -- representation-analysis probe battery, all 7 models\n"
                 "(blue = context-pooled reconstruction backbones, orange = patch-independent models)",
                 fontsize=10)
    plt.tight_layout()
    out_a = f"{Config.RESULTS_DIR}/study_a_probe_battery_all7_{timestamp}.png"
    plt.savefig(out_a, dpi=140)
    plt.close(fig)
    print(f"OK {out_a}")

    # ---- Study B: 2-panel bar chart ----
    names_b = [n for n in Config.MODEL_ORDER if n in all_results_b]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    ax = axes[0]
    xw = np.arange(len(names_b))
    indomain_vals = [all_results_b[n]['indomain']['balanced_acc'] for n in names_b]
    pcam_vals = [all_results_b[n]['pcam']['balanced_acc'] for n in names_b]
    ax.bar(xw - 0.2, indomain_vals, width=0.4, label='in-domain (Camelyon17)')
    ax.bar(xw + 0.2, pcam_vals, width=0.4, label='cross-dataset (PCam)')
    ax.axhline(50.0, ls='--', c='grey', label='chance (50%)')
    ax.set_xticks(xw); ax.set_xticklabels(names_b, rotation=35, ha='right')
    ax.set_ylabel('balanced accuracy (%)')
    ax.set_title('Tumor probe: in-domain vs. cross-dataset transfer')
    ax.legend(fontsize=8); ax.grid(alpha=.3, axis='y')

    ax = axes[1]
    gaps = [all_results_b[n]['gap_balanced_acc'] for n in names_b]
    bar_colors = ['#4C72B0' if g >= 0 else '#DD8452' for g in gaps]
    ax.bar(names_b, gaps, color=bar_colors)
    ax.axhline(0.0, ls='-', c='black', lw=0.8)
    ax.set_ylabel('generalization gap (pts)')
    ax.set_title('In-domain minus cross-dataset\n(smaller = better transfer)')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(alpha=.3, axis='y')
    plt.tight_layout()
    out_b = f"{Config.RESULTS_DIR}/study_b_cross_dataset_all7_{timestamp}.png"
    plt.savefig(out_b, dpi=140)
    plt.close(fig)
    print(f"OK {out_b}")


def save_combined_csvs(all_results_a, all_results_b, timestamp):
    import pandas as pd
    rows_a = []
    for name in Config.MODEL_ORDER:
        if name not in all_results_a:
            continue
        r = dict(all_results_a[name])
        r['model'] = name
        r['representation'] = "context-pooled (window)" if name in Config.WINDOWED_MODELS else "patch-mean-pooled (independent)"
        rows_a.append(r)
    if rows_a:
        path_a = f"{Config.RESULTS_DIR}/study_a_probe_battery_all7_{timestamp}.csv"
        pd.DataFrame(rows_a).to_csv(path_a, index=False)
        print(f"OK {path_a}")

    rows_b = []
    for name in Config.MODEL_ORDER:
        if name not in all_results_b:
            continue
        r = all_results_b[name]
        rows_b.append({
            'model': name,
            'indomain_acc': r['indomain']['acc'], 'indomain_balanced_acc': r['indomain']['balanced_acc'],
            'indomain_auroc': r['indomain']['auroc'], 'indomain_auroc_reliable': r['indomain']['auroc_reliable'],
            'pcam_acc': r['pcam']['acc'], 'pcam_balanced_acc': r['pcam']['balanced_acc'],
            'pcam_auroc': r['pcam']['auroc'], 'pcam_auroc_reliable': r['pcam']['auroc_reliable'],
            'gap_balanced_acc': r['gap_balanced_acc'],
        })
    if rows_b:
        path_b = f"{Config.RESULTS_DIR}/study_b_cross_dataset_all7_{timestamp}.csv"
        pd.DataFrame(rows_b).to_csv(path_b, index=False)
        print(f"OK {path_b}")


# ====================================================================
# MAIN
# ====================================================================
def main():
    from datetime import datetime
    set_seed(Config.SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("=" * 70)
    print("PROBE STUDIES -- FULL, COMBINED, ALL 7 MODELS")
    print("=" * 70)
    print(f"device: {device}")
    print(f"models: {Config.MODEL_ORDER}")
    print("=" * 70)

    # ---- resolve recon checkpoints once (shared by all 3 windowed models) ----
    recon_root = _resolve_recon_checkpoint_dir()
    recon_ckpts = _find_recon_checkpoints(recon_root)
    missing_recon = [n for n in Config.WINDOWED_MODELS if n not in recon_ckpts]
    if missing_recon:
        print(f"FLAG missing recon checkpoints for {missing_recon} -- these will be SKIPPED.")

    # ---- datasets (loaded once, shared by every model) ----
    print("\nLoading Camelyon17-WILDS...")
    cam17 = load_camelyon17()
    cam17_label_col = resolve_label_column(cam17, "Camelyon17-WILDS", Config.CAMELYON17_LABEL_COLUMN)

    print("\nLoading PatchCamelyon (Study B target domain)...")
    pcam = load_pcam()
    pcam_label_col = resolve_label_column(pcam, "PatchCamelyon", Config.PCAM_LABEL_COLUMN)

    print("\nBuilding spatial index (3x3 lattices, shared by every model in Study A)...")
    index = SpatialIndex(cam17, Config.GRID_SIZE,
                         cache_path=f"{Config.CACHE_DIR}/probe_lattices_{Config.GRID_SIZE}x{Config.GRID_SIZE}.pkl")

    print("\nBuilding Camelyon17-WILDS patient-level split (for Study B's in-domain baseline)...")
    cam17_meta = cam17.select_columns(['patient', 'center', cam17_label_col]).to_pandas()
    train_pats_by_centre, test_pats_by_centre = build_patient_split_by_centre(
        cam17_meta, Config.VAL_PATIENT_FRAC, Config.SEED)
    train_mask = np.zeros(len(cam17_meta), dtype=bool)
    test_mask = np.zeros(len(cam17_meta), dtype=bool)
    for centre in Config.ALL_CENTERS:
        c_mask = (cam17_meta['center'] == centre).to_numpy()
        pat_arr = cam17_meta['patient'].to_numpy()
        train_mask |= c_mask & np.isin(pat_arr, list(train_pats_by_centre.get(centre, set())))
        test_mask |= c_mask & np.isin(pat_arr, list(test_pats_by_centre.get(centre, set())))
    cam17_labels = cam17_meta[cam17_label_col].to_numpy().astype(np.int64)
    train_pool_idx = np.where(train_mask)[0]
    test_pool_idx = np.where(test_mask)[0]
    train_sel = train_pool_idx[stratified_sample_indices(
        cam17_labels[train_pool_idx], Config.CAMELYON17_TRAIN_N_PER_CLASS, Config.SEED)]
    test_sel = test_pool_idx[stratified_sample_indices(
        cam17_labels[test_pool_idx], Config.CAMELYON17_TEST_N_PER_CLASS, Config.SEED + 1)]
    print(f"  Study B probe-train sample: {len(train_sel):,} patches   "
          f"in-domain test sample: {len(test_sel):,} patches")

    print("\nSampling PatchCamelyon target-domain evaluation set...")
    pcam_labels = np.array(pcam[pcam_label_col], dtype=np.int64)
    pcam_sel = stratified_sample_indices(pcam_labels, Config.PCAM_TEST_N_PER_CLASS, Config.SEED + 2)
    print(f"  Study B cross-dataset test sample: {len(pcam_sel):,} patches")

    all_results_a, all_results_b = {}, {}
    for name in Config.MODEL_ORDER:
        if name in Config.WINDOWED_MODELS and name not in recon_ckpts:
            print(f"\nSKIPPING {name} (checkpoint not found)")
            continue
        try:
            model = load_model_for_probes(name, device, recon_ckpts)
        except Exception as e:
            print(f"\nFLAG could not load {name}: {e} -- SKIPPING both studies for this model.")
            continue

        try:
            all_results_a[name] = run_study_a_for_model(name, model, cam17, index, device, cam17_label_col)
        except Exception as e:
            print(f"FLAG Study A failed for {name}: {e}")

        try:
            all_results_b[name] = run_study_b_for_model(
                name, model, cam17, pcam, device, cam17_label_col, pcam_label_col,
                train_sel, test_sel, pcam_sel)
        except Exception as e:
            print(f"FLAG Study B failed for {name}: {e}")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print_study_a_summary(all_results_a)
    print_study_b_summary(all_results_b)
    make_combined_figures(all_results_a, all_results_b, timestamp)
    save_combined_csvs(all_results_a, all_results_b, timestamp)

    print("\n" + "=" * 70)
    print("DONE.")
    print(f"   Combined figures + tables : {Config.RESULTS_DIR}")
    print("=" * 70)
    return all_results_a, all_results_b


if __name__ == '__main__':
    main()

PROBE STUDIES -- FULL, COMBINED, ALL 7 MODELS
device: cuda
models: ['recon_resnet18', 'recon_vit_small', 'recon_swin_tiny', 'baseline_stain_aug', 'kaiko_vits16', 'lunit_vits16', 'phikon']
OK found recon checkpoints at classic input mount /kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints

Loading Camelyon17-WILDS...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

...

  centre 1: 8 patients (6 train / 2 test) -> 9450 train patches, 4050 test patches


  centre 2: 8 patients (6 train / 2 test) -> 9450 train patches, 4050 test patches


  centre 3: 7 patients (5 train / 2 test) -> 9450 train patches, 4050 test patches


  centre 4: 7 patients (5 train / 2 test) -> 9450 train patches, 4050 test patches
  [Probe 2 -- centre/deep]  logreg  97.20%   mlp  98.71%   (chance 20.00%)
  [Probe 3 -- centre/color] acc  87.52%   (chance 20.00%)  -- no model, pixel stats only
  [Probe 4 -- position]     acc  11.63%   (chance 11.11%)
  [Probe 5 -- patient ID]   acc  99.99%   (chance 3.33%, 30 patients, closed-set)

STUDY B -- phikon  (cross-dataset: Camelyon17-WILDS -> PatchCamelyon)


  [in-domain -- Camelyon17->Camelyon17]   acc  86.95%  balanced-acc  86.95%  auroc  0.9635  (261/2000 errors, 1000 pos/1000 neg)
  [cross-dataset -- Camelyon17->PCam]     acc  82.15%  balanced-acc  82.15%  auroc  0.9276  (1071/6000 errors, 3000 pos/3000 neg)
  [generalization gap, balanced-acc]      +4.80 pts (smaller = better transfer)

STUDY A SUMMARY -- centre(deep) / centre(colour) / position / patient, all 7 models
(tumor-vs-normal probe stays disabled for every model -- see module docstring)
model                representation            centre(deep)  centre(color)   position    patient
recon_resnet18       context-pooled (window)         28.53%         87.52%     99.73%     31.03%
recon_vit_small      context-pooled (window)         43.96%         87.52%     99.15%     42.93%
recon_swin_tiny      context-pooled (window)         38.13%         87.52%     99.70%     50.53%
baseline_stain_aug   patch-mean-pooled (indep.)        36.00%         87.52%     13.09%     95.10%
kaiko_vits